In [ ]:
import numpy as np
import sys
import os
import matplotlib.pyplot as plt

# Define the path to the src directory manually
src_path = os.path.abspath('../src')
sys.path.insert(0, src_path)

from module_tidal import TidalData
from module_rotor import RotorData
from module_rotor_simulation import RotorSimulation
from module_vessel import VesselData
from module_lcoe import LCOE
from module_constraint_checker import ConstraintChecker
from module_battery_charging import BatteryCharging


from constUnitConvert import ConstantsUnitConversion
from constGlobal import ConstantsGlobal

CONVERT = ConstantsUnitConversion()
GLOBAL = ConstantsGlobal()


# Use color-blind friendly style
plt.style.use('tableau-colorblind10')

In [ ]:

# Define tidal data parameters
station = "PCT3716"  # Site 2
startdate = 20200201  # yyyyMMdd
rangeHr = 2 * 7 * 24  # Two weeks
timestep = 5.0  # Timestep in seconds


In [ ]:

# Initialize TidalData class
tidal_data = TidalData(station, startdate, rangeHr, timestep)
flow_speeds, times, dmoor_m, lat_rad, lon_rad, station_name, nearest_city, cable_length = tidal_data.load_tidal_data("../data/AlaskaCityLatLong.txt")


In [ ]:

plt.plot(times,flow_speeds)
print(dmoor_m)
print(lat_rad)
print(lon_rad)
print(station_name)
print(nearest_city)
print(cable_length*CONVERT.m2mile)
print(cable_length)

In [ ]:

# Define simulation parameters
config = {
    'Radius': 1.0,
    'Prated': 1000.0,
    'dCable': 10.0,
    'dMoor': dmoor_m,
    'Uinf': flow_speeds,
    't': times,
    'CpFunc': None,  # Placeholder, will be set later
    'CqFunc': None,  # Placeholder, will be set later
    'CtFunc': None,  # Placeholder, will be set later
    'CpOpt': None,  # Placeholder, will be set later
    'TSROpt': None,  # Placeholder, will be set later
    'Umin': 0.0,
    'withBrake': True,
    'control_strategy': 'constant_speed',
    'attachment_method': 'cable'
}

# Initialize RotorData class
rotor_filename = "../data/Sitkana_rotor_data_blade_1.txt"
rotor_data = RotorData(rotor_filename)

# Set Cp, Cq, and Ct functions from RotorData class
config['CpFunc'] = rotor_data.get_cp
config['CqFunc'] = rotor_data.get_cq
config['CtFunc'] = rotor_data.get_ct

# Set CpOpt and TSROpt from RotorData class
config['CpOpt'] = rotor_data.CpOpt
config['TSROpt'] = rotor_data.TSROpt
config['TSRmax'] = rotor_data.TSRmax
config['efficiency'] = 0.9

config

rotor_data

In [ ]:
rotor_data.TSRmax
rotor_sim = RotorSimulation(config)


In [ ]:

possibleSpeed = rotor_sim.TSROpt * rotor_sim.Uinf / rotor_sim.Radius  # Assuming optimal TSR condition, what is the speed range
possibleSpeedRange = np.linspace(np.min(possibleSpeed), np.max(possibleSpeed), 50)  # Initial guess range
avePowerRange = -1 * np.array([rotor_sim.objectiveFunction_findOptimalConstantSpeed(ii, rotor_sim.Radius, rotor_sim.Uinf, rotor_sim.t, rotor_sim.CpFunc) for ii in possibleSpeedRange])
plt.plot(possibleSpeedRange,avePowerRange,'.')

maxspeed = np.argmax(rotor_sim.find_optimal_constant_speed())

positive_indices = np.where(avePowerRange > 0)[0]
min_Speed = possibleSpeedRange[positive_indices[0]]
max_Speed = possibleSpeedRange[positive_indices[-1]]

print(min_Speed)
print(max_Speed)
import scipy as sp
result = sp.optimize.minimize_scalar(rotor_sim.objectiveFunction_findOptimalConstantSpeed, bounds=(min_Speed, max_Speed), method='bounded', args=(rotor_sim.Radius, rotor_sim.Uinf, rotor_sim.t, rotor_sim.CpFunc))
print(result)

idx = np.argmax(avePowerRange)
possibleSpeedRange[idx]



In [ ]:
rotor_sim.simulate()
result = rotor_sim.get_results()

In [ ]:
result

In [ ]:
plt.subplot(3,1,1)
plt.plot(result['Pelec'])

plt.subplot(3,1,2)
plt.plot(result['w'],'-') 
plt.title('Speed [rad/s]')

plt.subplot(3,1,3)
plt.plot(result['Tc'],'-') 
plt.title('Torque [Nm]')

print(f'max speed (RPM): {np.max(result['w']*CONVERT.rads2rpm)}')
print(f'mean speed (RPM): {np.average(result['w']*CONVERT.rads2rpm)}')
print(f'min speed (RPM): {np.min(result['w']*CONVERT.rads2rpm)}')

print(f'max torque (N): {np.max(result['Tc'])}')
print(f'mean torque (N): {np.average(result['Tc'])}')
print(f'min torque (N): {np.min(result['Tc'])}')


In [ ]:

plt.figure()
plt.plot(result['w']*CONVERT.rads2rpm,result['Tc'])
plt.xlabel('Speed [RPM]')
plt.ylabel('Torque [N]')

# Ship Design

In [ ]:
number_of_turbines = 1
turbulence_intensity = 0.0
discount_rate = 0.01
lifetime = 20
BatteryCapacity_kWh = 10

# Test user-defined vessel properties (Fishing vessel)
user_vessel_properties = {
    'Xm': 5.77,
    'Zm': 1.65,
    'Kphi': 1.95e6,
    'theta': 45.0 * np.pi / 180.0,
    'phi': 10.0 * np.pi / 180.0,
    'area': 2.25,
    'Cd': 1.0
}

# initialize classes
vessel = VesselData(
    user_defined=True,
    vessel_properties=user_vessel_properties
)

vessel.print_all_attributes()

In [ ]:
result['Ft']
constraint_checker = ConstraintChecker(config['Radius'], rotor_data.get_cpmin)

constraint_checker.check_pitch_constraint(vessel, result['Uinf_adjusted'], result['Ft'], result['dHub'], number_of_turbines)

In [ ]:
Vessel_Drag_Force = vessel.calculate_vessel_drag_force(result['Uinf_adjusted'])
print(Vessel_Drag_Force)

In [ ]:
constraint_checker = ConstraintChecker(config['Radius'], rotor_data.get_cpmin)

if constraint_checker.check_depth_constraint(result['dHub']):
    print("Depth constraint is satisfied")

if constraint_checker.check_cavitation_constraint(result['TSR'], result['Uinf_adjusted'], result['w'], result['dHub']):
    print("Cavitation constraint is satisfied")

if constraint_checker.check_pitch_stiffness_constraint(vessel):
    print("Pitch stiff constraint is satisfied")
        
if constraint_checker.check_pitch_constraint(vessel, result['Uinf_adjusted'], result['Ft'],result['dHub'], number_of_turbines):
    print("Pitch stiff constraint is satisfied")

In [ ]:
lcoe_calculator = LCOE(
    turbine_radius=config['Radius'], 
    turbine_rated_power=config['Prated'], 
    number_of_turbines=number_of_turbines, 
    hub_depth=result['dHub'], 
    lifetime=lifetime,
    discount_rate=discount_rate,
    turbulence_intensity=turbulence_intensity,
    customer='customer_B', 
    application='battery_charging'
)


# Calculate total CAPEX
lcoe_calculator.set_instantaneous_power(result['Pelec'], result['t'])

total_capex = lcoe_calculator.calculate_total_capex(cable_length, 
                                                   dmoor_m, 
                                                   Vessel_Drag_Force, 
                                                   result['Ft'], 
                                                   vessel.VesselVolume, 
                                                   BatteryCapacity_kWh)

In [ ]:
print(f"Total CAPEX: {total_capex[0]:.2f} USD")

annual_energy = lcoe_calculator.calculate_annual_energy()
print(f"Annual Energy: {annual_energy:.2f} kWh")

capacity_factor = lcoe_calculator.calculate_capacity_factor()
print(f"Capacity Factor: {capacity_factor:.2%}")


myLCOE = lcoe_calculator.calculate_lcoe(cable_length, 
                                                   dmoor_m, 
                                                   Vessel_Drag_Force, 
                                                   result['Ft'], 
                                                   vessel.VesselVolume, 
                                                   BatteryCapacity_kWh)
print(f"LCOE: {myLCOE:.2f}$/kWh")

In [ ]:

battery_charging = BatteryCharging(BatteryCapacity_kWh, number_of_turbines, turbulence_intensity)
num_batteries_charged, charge_times_hr = battery_charging.chargeBattery_continuous(result['Pelec'], result['t'], visualise=False)
charge_time_average = np.average(charge_times_hr)

In [ ]:
charge_time_average